In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import datetime
from pathlib import Path

import dotenv

dotenv.load_dotenv("tokens.env")

curr_date: str = datetime.datetime.now().isoformat().split("T")[0]

print(curr_date)

/mnt/sdd1/atharvas/formulacode/datasmith
2025-09-20


In [2]:
import docker
from tqdm.auto import tqdm

all_images = [
    line.split()[10]
    for line in Path("scratch/scripts/parallel_validate_containers.log").read_text().splitlines()
    if "buildx build" in line
]
images_to_remove = [
    line.split()[3]
    for line in Path("scratch/scripts/parallel_validate_containers.log").read_text().splitlines()
    if "failed to run" in line
]
images_to_keep = list(set(all_images) - set(images_to_remove))
print(f"Keeping {len(images_to_keep)} images, removing {len(images_to_remove)} images")
# Increase timeout significantly for large image operations
client = docker.from_env(timeout=3600)  # 1 hour timeout
for img in tqdm(images_to_remove):
    try:
        client.images.remove(img, force=True)
    except Exception:
        continue

/mnt/sdd1/atharvas/formulacode/datasmith/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Keeping 381 images, removing 92 images


100%|██████████| 92/92 [00:00<00:00, 439.35it/s]


In [4]:
len(images_to_keep)

381

In [3]:
from docker.models.images import Image


def get_image(image_name: str) -> Image | None:
    client = docker.from_env(timeout=3600)  # 1 hour timeout
    try:
        return client.images.get(image_name)
    except Exception:
        return None


images = {img_name.split(":")[0]: get_image(img_name) for img_name in images_to_keep}
images = {img_name: img for img_name, img in images.items() if img is not None}
print(f"Found {len(images)} images locally")

Found 80 images locally


In [6]:
import contextlib
import json
import os
import re
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config as BotoConfig


def _safe_tar_name(image_name: str) -> str:
    """
    Make a filesystem- and S3-friendly tar name from the image name.
    e.g. 'registry:5000/ns/app:1.2.3' -> 'registry_5000_ns_app_1.2.3.tar.gz'
    """
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", image_name.replace("/", "_").replace(":", "_"))
    return f"{base}.tar.gz"


def _create_tarball_subprocess(image_name: str, tar_gz_path: str, max_retries: int = 3) -> None:
    """
    Create a tarball using subprocess calls to docker save and gzip.
    This is much more memory efficient and faster than Python-based streaming.
    """
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(tar_gz_path), exist_ok=True)

    temp_tar_path = tar_gz_path.replace(".tar.gz", ".tar")

    for attempt in range(max_retries):
        try:
            print(f"[+] Attempt {attempt + 1}/{max_retries}: Creating tarball for {image_name}")

            # Use docker save to create tar file
            docker_cmd = ["docker", "save", image_name, "-o", temp_tar_path]
            result = subprocess.run(docker_cmd, capture_output=True, text=True, check=True)

            print(f"[+] Compressing {temp_tar_path} to {tar_gz_path}")
            # Use gzip to compress the tar file
            gzip_cmd = ["gzip", "-c", temp_tar_path]
            with open(tar_gz_path, "wb") as f_out:
                subprocess.run(gzip_cmd, stdout=f_out, check=True)

            # Remove the temporary tar file
            os.remove(temp_tar_path)
            print(f"[✓] Successfully created tarball for {image_name}")
            return

        except subprocess.CalledProcessError as e:
            print(f"[!] Attempt {attempt + 1} failed for {image_name}: {e}")
            if e.stderr:
                print(f"[!] Error output: {e.stderr}")
        except Exception as e:
            print(f"[!] Attempt {attempt + 1} failed for {image_name}: {e}")

        # Clean up any partial files
        for path in [temp_tar_path, tar_gz_path]:
            if os.path.exists(path):
                with contextlib.suppress(Exception):
                    os.remove(path)

        if attempt < max_retries - 1:
            print(f"[+] Retrying {image_name} in 5 seconds...")
            import time

            time.sleep(5)

    raise RuntimeError(f"Failed to create tarball for {image_name} after {max_retries} attempts")


def _upload_single_file(
    s3_client, local_path: str, bucket: str, key: str, extra_args: dict, tx_config: TransferConfig
) -> str:
    """Upload a single file to S3."""
    try:
        print(f"[+] Uploading {local_path} → s3://{bucket}/{key}")
        s3_client.upload_file(local_path, bucket, key, ExtraArgs=extra_args, Config=tx_config)
        uri = f"s3://{bucket}/{key}"
        print(f"[✓] Uploaded: {uri}")
        return uri
    except Exception as e:
        print(f"[!] Error uploading {local_path}: {e}")
        raise


def upload_images_to_s3(
    images: dict[str, "Image"],
    bucket: str,
    prefix: str = "",
    region: str | None = None,
    sse: str | None = None,  # e.g. "AES256" or "aws:kms"
    tarball_dir: str = "scratch/artifacts/tarballs",
    force: bool = False,
    max_workers_tarball: int = 4,
    max_workers_upload: int = 8,
) -> dict[str, str]:
    """
    For each {name: Image} entry, produce a .tar.gz and upload to s3://bucket/prefix/<name>.tar.gz
    Also saves the tarballs locally to tarball_dir.
    If force is enabled, the local tarball and uploaded tarball are overwritten.

    Args:
        max_workers_tarball: Number of parallel workers for tarball creation
        max_workers_upload: Number of parallel workers for S3 uploads
    """
    # Prepare S3 client and config
    session = boto3.session.Session(region_name=region)
    s3 = session.client("s3", config=BotoConfig(retries={"max_attempts": 10}))
    tx_config = TransferConfig(multipart_threshold=8 * 1024 * 1024, multipart_chunksize=8 * 1024 * 1024)

    extra_args = {"ContentType": "application/gzip"}
    if sse:
        extra_args["ServerSideEncryption"] = sse

    # Stage 1: Create all tarballs in parallel
    print(f"[+] Stage 1: Creating {len(images)} tarballs with {max_workers_tarball} workers")
    tarball_tasks = []

    for name, img in images.items():
        tar_gz_name = _safe_tar_name(name)
        local_tar_gz_path = os.path.join(tarball_dir, tar_gz_name)

        if os.path.exists(local_tar_gz_path) and not force:
            print(f"[+] Skipping {name} as it already exists locally")
            continue

        tarball_tasks.append((name, local_tar_gz_path))

    # Create tarballs in parallel
    with ThreadPoolExecutor(max_workers=max_workers_tarball) as executor:
        future_to_name = {executor.submit(_create_tarball_subprocess, name, path): name for name, path in tarball_tasks}

        for future in as_completed(future_to_name):
            name = future_to_name[future]
            try:
                future.result()
            except Exception as e:
                print(f"[!] Failed to create tarball for {name}: {e}")
                # Remove from tasks if it failed
                tarball_tasks = [(n, p) for n, p in tarball_tasks if n != name]

    # Stage 2: Upload all tarballs to S3 in parallel
    print(f"[+] Stage 2: Uploading {len(tarball_tasks)} tarballs with {max_workers_upload} workers")
    mapping: dict[str, str] = {}

    upload_tasks = []
    for name, local_tar_gz_path in tarball_tasks:
        if os.path.exists(local_tar_gz_path):
            tar_gz_name = _safe_tar_name(name)
            key = f"{prefix.strip('/')}/{tar_gz_name}" if prefix else tar_gz_name
            upload_tasks.append((name, local_tar_gz_path, key))

    # # Upload to S3 in parallel
    # with ThreadPoolExecutor(max_workers=max_workers_upload) as executor:
    #     future_to_name = {
    #         executor.submit(_upload_single_file, s3, local_path, bucket, key, extra_args, tx_config): name
    #         for name, local_path, key in upload_tasks
    #     }

    #     for future in as_completed(future_to_name):
    #         name = future_to_name[future]
    #         try:
    #             uri = future.result()
    #             mapping[name] = uri
    #         except Exception as e:
    #             print(f"[!] Failed to upload {name}: {e}")
    #             # Clean up the local file if upload failed
    #             local_path = next((path for n, path, _ in upload_tasks if n == name), None)
    #             if local_path and os.path.exists(local_path):
    #                 with contextlib.suppress(Exception):
    #                     os.remove(local_path)

    return mapping


# Filter out None values from images dict
valid_images = {name: img for name, img in images.items() if img is not None}

mapping = upload_images_to_s3(
    valid_images,
    bucket=os.environ["AWS_S3_BUCKET"],
    prefix="docker-tarballs/",
    region=os.environ.get("AWS_REGION", None),
    max_workers_tarball=30,  # Adjust based on your system
    max_workers_upload=10,  # Adjust based on your network bandwidth
)

Path("scratch/artifacts/aws_docker_image_s3_mapping.json").write_text(json.dumps(mapping, indent=2))

[+] Stage 1: Creating 80 tarballs with 30 workers
[+] Skipping scikit-learn-scikit-learn-432778464cbffc8ca675c1df786c31f8c23fc62c as it already exists locally
[+] Skipping mie-lab-trackintel-c4573e0ca8360df74995875db67aa91d6a2ea6bb as it already exists locally
[+] Skipping dedupeio-dedupe-9f8e6f77490f9686cf4e69f1b6a1dd2055e76e05 as it already exists locally
[+] Skipping scipy-scipy-ff417e189d353f2cc791b66e6fff33c18f6ba85d as it already exists locally
[+] Skipping textualize-rich-ef0f5ca24b191633f796b31fc958244d26eb1259 as it already exists locally
[+] Skipping xitorch-xitorch-a79a194160b34afabd4a1e920c1d73fd84b91ec4 as it already exists locally
[+] Skipping textualize-rich-5b3f9bbfa88b384d3b161ca9884bb89fbcf46a13 as it already exists locally
[+] Skipping scikit-learn-scikit-learn-281e7e3d8e9cbf4d493efd0aa4e0e0d4d5848191 as it already exists locally
[+] Skipping scipy-scipy-3f81d06a530184c3f191499a8a84fb80658a92d8 as it already exists locally
[+] Skipping scipy-scipy-b44a16ccf0787819faa

2

In [ ]:
import gzip
import json
import os
import shutil

from boto3.s3.transfer import TransferConfig


def _safe_tar_name(image_name: str) -> str:
    """
    Make a filesystem- and S3-friendly tar name from the image name.
    e.g. 'registry:5000/ns/app:1.2.3' -> 'registry_5000_ns_app_1.2.3.tar.gz'
    """
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", image_name.replace("/", "_").replace(":", "_"))
    return f"{base}.tar.gz"


def _save_image_to_tar_gz(img, tar_gz_path: str, max_retries: int = 3) -> None:
    """
    Stream-save a docker image to tar.gz without loading it all into memory.
    Includes retry logic for timeout issues.
    """
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(tar_gz_path), exist_ok=True)

    temp_tar_path = tar_gz_path.replace(".tar.gz", ".tar")

    for attempt in range(max_retries):
        try:
            print(f"[+] Attempt {attempt + 1}/{max_retries}: Saving image to {temp_tar_path}")

            # image.save(named=True) yields a generator of bytes
            stream = img.save(named=True)

            # Write to a temporary tar file first, then compress
            with open(temp_tar_path, "wb") as f:
                for chunk in stream:
                    f.write(chunk)

            print(f"[+] Compressing {temp_tar_path} to {tar_gz_path}")
            # Compress the tar file to tar.gz
            with open(temp_tar_path, "rb") as f_in, gzip.open(tar_gz_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

            # Remove the temporary tar file
            os.remove(temp_tar_path)
            print("[✓] Successfully saved and compressed image")
        except Exception as e:
            print(f"[!] Attempt {attempt + 1} failed: {e}")
            # Clean up any partial files
            for path in [temp_tar_path, tar_gz_path]:
                if os.path.exists(path):
                    with contextlib.suppress(Exception):
                        os.remove(path)

            if attempt == max_retries - 1:
                raise
            else:
                print("[+] Retrying in 5 seconds...")
                import time

                time.sleep(5)
        else:
            return


def upload_images_to_s3(
    images: dict[str, "Image"],
    bucket: str,
    prefix: str = "",
    region: str | None = None,
    sse: str | None = None,  # e.g. "AES256" or "aws:kms"
    tarball_dir: str = "scratch/artifacts/tarballs",
    force: bool = False,
) -> dict[str, str]:
    """
    For each {name: Image} entry, produce a .tar.gz and upload to s3://bucket/prefix/<name>.tar.gz
    Also saves the tarballs locally to tarball_dir.
    If force is enabled, the local tarball and uploaded tarball are overwritten.
    """
    session = boto3.session.Session(region_name=region)
    s3 = session.client("s3", config=BotoConfig(retries={"max_attempts": 10}))
    tx_config = TransferConfig(multipart_threshold=8 * 1024 * 1024, multipart_chunksize=8 * 1024 * 1024)

    mapping: dict[str, str] = {}

    for name, img in images.items():
        tar_gz_name = _safe_tar_name(name)
        key = f"{prefix.strip('/')}/{tar_gz_name}" if prefix else tar_gz_name

        # Local path for the tarball
        local_tar_gz_path = os.path.join(tarball_dir, tar_gz_name)
        if os.path.exists(local_tar_gz_path) and not force:
            print(f"[+] Skipping {name} as it already exists locally")
            continue

        try:
            print(f"[+] Saving image '{name}' → {local_tar_gz_path}")
            _save_image_to_tar_gz(img, local_tar_gz_path)

            extra = {"ContentType": "application/gzip"}
            if sse:
                extra["ServerSideEncryption"] = sse

            print(f"[+] Uploading {local_tar_gz_path} → s3://{bucket}/{key}")
            s3.upload_file(local_tar_gz_path, bucket, key, ExtraArgs=extra, Config=tx_config)

            uri = f"s3://{bucket}/{key}"
            mapping[name] = uri
            print(f"[✓] Uploaded: {uri}")
        except Exception as e:
            print(f"[!] Error processing {name}: {e}")
            # Clean up the local file if it was created but upload failed
            if os.path.exists(local_tar_gz_path):
                with contextlib.suppress(Exception):
                    os.remove(local_tar_gz_path)

    return mapping


mapping = upload_images_to_s3(
    images,
    bucket=os.environ["AWS_S3_BUCKET"],
    prefix="docker-tarballs/",
    region=os.environ.get("AWS_REGION", None),
)

Path("Scratch/artifacts/aws_docker_image_s3_mapping.json").write_text(json.dumps(mapping, indent=2))

In [ ]:
# #!/usr/bin/env bash
# set -euo pipefail

# S3_URI="s3://my-artifacts-bucket/bundles/docker-images-2025-09-19.tar.gz"
# WORKDIR="${WORKDIR:-/tmp/docker-image-bundle}"

# mkdir -p "$WORKDIR"
# cd "$WORKDIR"

# echo "Downloading bundle..."
# aws s3 cp "$S3_URI" ./bundle.tar.gz

# echo "Extracting bundle..."
# tar -xzf bundle.tar.gz

# echo "Loading images into Docker..."
# shopt -s nullglob
# for img_tar in *.tar; do
#   echo "Loading $img_tar ..."
#   docker load -i "$img_tar"
# done

# echo "Done. Loaded images:"
# docker images
